# Fine-tuning de Demucs v4 (htdemucs) con MUSDB18

Ajuste fino del modelo **htdemucs** que usa el sistema de karaoke para
separar la voz del acompañamiento. Se parte de los pesos preentrenados
publicados por Meta y se sigue entrenando con el dataset **MUSDB18**
(fragmentos de 7 s, licencia CC BY 4.0), el mismo que está en
`pretrained_models/demucs/dataset/musdb18-7s`.

Pasos:
1. Preparar el entorno (GPU de Google Colab).
2. Descargar el dataset y verificar su checksum.
3. Medir el SDR del modelo preentrenado sobre el conjunto de prueba.
4. Entrenar (fine-tuning) con el conjunto de entrenamiento.
5. Medir de nuevo el SDR y comparar.
6. Guardar el modelo afinado en el formato de Demucs.

En Colab: **Entorno de ejecución > Cambiar tipo de entorno > GPU**.

## 1. Entorno

In [ ]:
!nvidia-smi -L
!pip install -q demucs==4.0.1

In [ ]:
import hashlib
import json
import os
import random
import subprocess
import time
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from demucs.apply import apply_model
from demucs.pretrained import get_model
from demucs.states import save_with_checksum, serialize_model
from omegaconf import OmegaConf

# Los archivos .th de Demucs guardan la clase del modelo; torch >= 2.6 los
# rechaza por defecto al cargarlos desde una carpeta local
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO, "| torch", torch.__version__)

## 2. Configuración del entrenamiento

In [ ]:
CONFIG = {
    "modelo_base": "htdemucs",
    "dataset": "MUSDB18 (fragmentos de 7 s)",
    "epocas": int(os.getenv("EPOCAS", "20")),
    "lote": int(os.getenv("LOTE", "4")),
    "segundos_por_ejemplo": 5.0,
    "tasa_aprendizaje": 3e-5,
    "perdida": "L1 sobre las 4 fuentes",
    "semilla": 42,
    # 0 = todas las pistas del conjunto
    "pistas_entrenamiento": int(os.getenv("PISTAS_ENTRENAMIENTO", "0")),
    "pistas_prueba": int(os.getenv("PISTAS_PRUEBA", "0")),
}
DIR_DATASET = Path(os.getenv("DIR_DATASET", "musdb18-7s"))
DIR_SALIDA = Path(os.getenv("DIR_SALIDA", "modelo_afinado"))
DIR_SALIDA.mkdir(exist_ok=True)

random.seed(CONFIG["semilla"])
np.random.seed(CONFIG["semilla"])
torch.manual_seed(CONFIG["semilla"])
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 3. Dataset MUSDB18

Se descarga de Zenodo (registro 3270814) y se comprueba el MD5. Es el mismo
archivo guardado en el proyecto; también puede subirse desde Google Drive y
apuntar `DIR_DATASET` a esa carpeta.

In [ ]:
URL = (
    "https://zenodo.org/records/3270814/files/"
    "MUSDB18-7-STEMS.zip?download=1"
)
MD5 = "dc2bdfac5b46ce742bd60009866758a5"

if not (DIR_DATASET / "train").is_dir():
    zip_dataset = Path("MUSDB18-7-STEMS.zip")
    if not zip_dataset.exists():
        print("Descargando el dataset...")
        urllib.request.urlretrieve(URL, zip_dataset)
    md5 = hashlib.md5(zip_dataset.read_bytes()).hexdigest()
    assert md5 == MD5, f"Checksum incorrecto: {md5}"
    with zipfile.ZipFile(zip_dataset) as z:
        z.extractall(DIR_DATASET)

for subconjunto in ("train", "test"):
    n = len(list((DIR_DATASET / subconjunto).glob("*.stem.mp4")))
    print(f"{subconjunto}: {n} canciones")

Cada archivo `.stem.mp4` tiene 5 pistas de audio: mezcla, batería, bajo,
otros y voz. Se decodifican con ffmpeg a 44,1 kHz estéreo.

In [ ]:
MODELO_BASE = get_model(CONFIG["modelo_base"]).models[0]
FUENTES = MODELO_BASE.sources  # ['drums', 'bass', 'other', 'vocals']
INDICE_STEM = {"drums": 1, "bass": 2, "other": 3, "vocals": 4}
TASA = MODELO_BASE.samplerate


def leer_stem(ruta, indice):
    datos = subprocess.run(
        [
            "ffmpeg", "-nostdin", "-v", "error", "-i", str(ruta),
            "-map", f"0:{indice}", "-f", "f32le", "-ac", "2",
            "-ar", str(TASA), "-",
        ],
        check=True,
        capture_output=True,
    ).stdout
    return np.frombuffer(datos, dtype=np.float32).reshape(-1, 2).T


def cargar(subconjunto, limite):
    rutas = sorted((DIR_DATASET / subconjunto).glob("*.stem.mp4"))
    if limite:
        rutas = rutas[:limite]
    pistas = []
    for ruta in rutas:
        fuentes = np.stack([leer_stem(ruta, INDICE_STEM[f]) for f in FUENTES])
        pistas.append((ruta.name[: -len(".stem.mp4")], torch.from_numpy(fuentes)))
    return pistas


entrenamiento = cargar("train", CONFIG["pistas_entrenamiento"])
prueba = cargar("test", CONFIG["pistas_prueba"])
print(len(entrenamiento), "pistas de entrenamiento,", len(prueba), "de prueba")

### Aumento de datos

Como en el entrenamiento original de Demucs: recorte aleatorio, ganancia
aleatoria por fuente, intercambio de canales, inversión de signo y
*remix* (cada fuente puede venir de otra canción). La mezcla de entrada es
la suma de las fuentes.

In [ ]:
class EjemplosMusdb(torch.utils.data.Dataset):
    def __init__(self, pistas, segundos, ejemplos_por_epoca):
        self.pistas = [fuentes for _, fuentes in pistas]
        self.largo = int(segundos * TASA)
        self.ejemplos = ejemplos_por_epoca

    def __len__(self):
        return self.ejemplos

    def recorte(self, fuentes):
        inicio = random.randint(0, fuentes.shape[-1] - self.largo)
        return fuentes[..., inicio : inicio + self.largo]

    def __getitem__(self, _):
        base = self.recorte(random.choice(self.pistas))
        if random.random() < 0.5:
            otras = [self.recorte(random.choice(self.pistas)) for _ in FUENTES]
            base = torch.stack([otras[i][i] for i in range(len(FUENTES))])
        ganancia = torch.empty(len(FUENTES), 1, 1).uniform_(0.25, 1.25)
        fuentes = base * ganancia
        if random.random() < 0.5:
            fuentes = fuentes.flip(1)
        if random.random() < 0.5:
            fuentes = -fuentes
        return fuentes.sum(0), fuentes

## 4. Métrica: SDR

Se usa el SDR de la competencia Music Demixing (MDX), en dB; más alto es
mejor. Se evalúa sobre la canción completa del conjunto de prueba.

In [ ]:
def sdr(referencia, estimacion, eps=1e-8):
    num = (referencia**2).sum(dim=(-2, -1)) + eps
    den = ((referencia - estimacion) ** 2).sum(dim=(-2, -1)) + eps
    return 10 * torch.log10(num / den)


@torch.no_grad()
def evaluar(modelo, pistas):
    modelo.eval()
    resultados = []
    for _, fuentes in pistas:
        mezcla = fuentes.sum(0)
        estimacion = apply_model(
            modelo, mezcla[None], device=DISPOSITIVO, split=True,
            overlap=0.25, progress=False,
        )[0].cpu()
        resultados.append(sdr(fuentes, estimacion))
    por_fuente = torch.stack(resultados).mean(0)
    return {f: round(float(v), 3) for f, v in zip(FUENTES, por_fuente)}


modelo = MODELO_BASE.to(DISPOSITIVO)
SEGMENTO_ORIGINAL = modelo.segment
inicio = time.time()
sdr_antes = evaluar(modelo, prueba)
print(f"SDR del modelo preentrenado ({time.time() - inicio:.0f} s):", sdr_antes)

## 5. Fine-tuning

In [ ]:
cargador = torch.utils.data.DataLoader(
    EjemplosMusdb(
        entrenamiento,
        CONFIG["segundos_por_ejemplo"],
        ejemplos_por_epoca=max(len(entrenamiento), CONFIG["lote"]) * 4,
    ),
    batch_size=CONFIG["lote"],
    shuffle=True,
    drop_last=True,
)
optimizador = torch.optim.Adam(
    modelo.parameters(), lr=CONFIG["tasa_aprendizaje"]
)
historial = []
for epoca in range(1, CONFIG["epocas"] + 1):
    modelo.train()
    perdidas = []
    inicio = time.time()
    for mezcla, fuentes in cargador:
        mezcla, fuentes = mezcla.to(DISPOSITIVO), fuentes.to(DISPOSITIVO)
        estimacion = modelo(mezcla)
        perdida = torch.nn.functional.l1_loss(estimacion, fuentes)
        optimizador.zero_grad()
        perdida.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 5.0)
        optimizador.step()
        perdidas.append(perdida.item())
    historial.append(float(np.mean(perdidas)))
    print(
        f"época {epoca:3d} | pérdida L1 {historial[-1]:.5f} "
        f"| {time.time() - inicio:.0f} s"
    )
# En entrenamiento HTDemucs ajusta su segmento al largo de los ejemplos; se
# restaura el original para la inferencia
modelo.segment = SEGMENTO_ORIGINAL

## 6. Resultados

In [ ]:
sdr_despues = evaluar(modelo, prueba)
print(f"{'fuente':8} {'antes':>8} {'después':>8} {'mejora':>8}")
for f in FUENTES:
    mejora = sdr_despues[f] - sdr_antes[f]
    print(f"{f:8} {sdr_antes[f]:8.3f} {sdr_despues[f]:8.3f} {mejora:+8.3f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(historial) + 1), historial, marker="o")
plt.xlabel("Época")
plt.ylabel("Pérdida L1")
plt.title("Fine-tuning de htdemucs con MUSDB18")
plt.grid(alpha=0.3)
plt.savefig(DIR_SALIDA / "perdida.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Guardar el modelo afinado

Se guarda en el formato de Demucs (`<firma>-<checksum>.th` más un `.yaml`),
para usarlo con `demucs -n htdemucs_karaoke --repo modelo_afinado`.

In [ ]:
for viejo in DIR_SALIDA.glob("karaokeft*.th"):
    viejo.unlink()
paquete = serialize_model(modelo.cpu(), OmegaConf.create(CONFIG), half=True)
save_with_checksum(paquete, DIR_SALIDA / "karaokeft.th")
(DIR_SALIDA / "htdemucs_karaoke.yaml").write_text("models: ['karaokeft']\n")
(DIR_SALIDA / "metricas.json").write_text(
    json.dumps(
        {
            "configuracion": CONFIG,
            "sdr_antes": sdr_antes,
            "sdr_despues": sdr_despues,
            "perdida_por_epoca": historial,
        },
        indent=2,
        ensure_ascii=False,
    )
)
print(sorted(p.name for p in DIR_SALIDA.iterdir()))

Prueba del modelo guardado: se carga desde la carpeta, igual que lo hace
el sistema con `--repo`, y separa la voz de una canción de prueba.

In [ ]:
guardado = get_model("htdemucs_karaoke", repo=DIR_SALIDA)
nombre, fuentes = prueba[0]
mezcla = fuentes.sum(0)
voz = apply_model(guardado, mezcla[None], device=DISPOSITIVO)[0]
voz = voz[FUENTES.index("vocals")].cpu()
sf.write(DIR_SALIDA / "ejemplo_mezcla.wav", mezcla.T.numpy(), TASA)
sf.write(DIR_SALIDA / "ejemplo_voz.wav", voz.T.numpy(), TASA)
print(nombre, "| SDR de la voz:", round(float(sdr(fuentes[3], voz)), 3))
try:
    from IPython.display import Audio, display

    display(Audio(str(DIR_SALIDA / "ejemplo_mezcla.wav")))
    display(Audio(str(DIR_SALIDA / "ejemplo_voz.wav")))
except ImportError:
    pass

In [ ]:
# Descarga los resultados (solo en Google Colab)
try:
    from google.colab import files

    subprocess.run(["zip", "-qr", "modelo_afinado.zip", str(DIR_SALIDA)])
    files.download("modelo_afinado.zip")
except ImportError:
    print("Resultados en", DIR_SALIDA.resolve())